In [1]:
# =============================================================================
# 10_download_daily_satellite_data.ipynb
#
# PURPOSE
# =============================================================================
#
# Create the MASTER acquisition-level Sentinel-1 and Sentinel-2 dataset for
# Hurricane Helene analysis.
#
#
# CURRENT SAMPLE
# =============================================================================
#
# Treatment:
#
#   first 10 treatment sites
#
# Counterfactual:
#
#   first 5 matched counterfactual sites for EACH selected treatment
#
# Maximum:
#
#   10 treatment sites
#   50 counterfactual sites
#   60 total sites
#
#
# STUDY PERIOD
# =============================================================================
#
# BEFORE:
#
#   2024-05-10 through 2024-09-26
#
# AFTER:
#
#   2024-09-27 through 2025-02-13
#
# TOTAL:
#
#   2024-05-10 through 2025-02-13
#
#
# IMPORTANT: "DAILY" DATA
# =============================================================================
#
# Daily means ACQUISITION-LEVEL imagery.
#
# A TIFF is created only when Sentinel-1 or Sentinel-2 actually observed
# the site.
#
# The notebook does NOT:
#
#   - fabricate observations for missing calendar dates
#   - interpolate satellite imagery
#   - average different dates
#
# A separate complete calendar is generated in:
#
#   daily_calendar_index.csv
#
#
# LOCAL-FIRST / RESUME-SAFE LOGIC
# =============================================================================
#
# For each expected acquisition date:
#
#   1. Construct expected TIFF path.
#
#   2. CHECK LOCAL FOLDER FIRST.
#
#   3. If TIFF already exists and passes validation:
#
#        - DO NOT build Earth Engine daily image
#        - DO NOT download
#        - calculate image quality locally
#        - status = existing
#        - download_action = skipped_existing
#
#   4. If TIFF exists but fails validation:
#
#        - rename to *.invalid
#        - create/download again
#
#   5. If TIFF does not exist:
#
#        - build same-day Earth Engine mosaic
#        - download
#        - validate
#
#
# PRIMARY QUALITY DEFINITION
# =============================================================================
#
# A spatial pixel is considered VALID if AT LEAST ONE output band has a
# finite value.
#
# Primary:
#
#   valid_pixel_fraction
#   valid_pixel_percentage
#
# Diagnostic:
#
#   valid_pixel_fraction_any_band
#   valid_pixel_fraction_all_bands
#
#
# SENTINEL-2 OUTPUT BANDS
# =============================================================================
#
# B2
# B3
# B4
# B8
# B11
# B12
# NDVI
# NDWI
#
#
# SENTINEL-1 OUTPUT BANDS
# =============================================================================
#
# VV
# VH
# VV_minus_VH
#
#
# OUTPUT STRUCTURE
# =============================================================================
#
# finals/
# └── daily_datasets/
#
#     ├── sentinel1/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── sentinel2/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── sentinel1_daily_inventory.csv
#     ├── sentinel2_daily_inventory.csv
#     ├── daily_satellite_inventory.csv
#     ├── daily_calendar_index.csv
#     ├── daily_dataset_summary.csv
#     ├── daily_folder_summary.csv
#     ├── daily_download_action_summary.csv
#     ├── selected_site_sample.csv
#     └── daily_failed_downloads.csv
#
# =============================================================================


# =============================================================================
# 1. Packages
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import time
import warnings

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="geemap",
)


print(
    "Packages loaded successfully."
)


# =============================================================================
# 2. Project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)


FINALS_DIR = (
    BASE_DIR /
    "finals"
)


DAILY_DIR = (
    FINALS_DIR /
    "daily_datasets"
)


S1_DAILY_DIR = (
    DAILY_DIR /
    "sentinel1"
)


S2_DAILY_DIR = (
    DAILY_DIR /
    "sentinel2"
)


TREATMENT_SITES_FILE = (
    FINALS_DIR /
    "treatment_sites.geojson"
)


COUNTERFACTUAL_SITES_FILE = (
    FINALS_DIR /
    "counterfactual_sites.geojson"
)


MATCHING_TABLE_FILE = (
    FINALS_DIR /
    "site_matching_table.csv"
)


DAILY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3. Study calendar
# =============================================================================

STUDY_START = pd.Timestamp(
    "2024-05-10"
)


STUDY_END = pd.Timestamp(
    "2025-02-13"
)


# Earth Engine filterDate uses exclusive end date.
EE_STUDY_END = (
    "2025-02-14"
)


HELENE_REFERENCE_DATE = pd.Timestamp(
    "2024-09-27"
)


BEFORE_START = pd.Timestamp(
    "2024-05-10"
)


BEFORE_END = pd.Timestamp(
    "2024-09-26"
)


AFTER_START = pd.Timestamp(
    "2024-09-27"
)


AFTER_END = pd.Timestamp(
    "2025-02-13"
)


def assign_period(
    date,
):

    date = pd.Timestamp(
        date
    )


    if date < HELENE_REFERENCE_DATE:

        return "before"


    return "after"


print(
    "\nStudy period:"
)


print(
    STUDY_START.strftime("%Y-%m-%d"),
    "through",
    STUDY_END.strftime("%Y-%m-%d"),
)


print(
    "\nBefore:"
)


print(
    BEFORE_START.strftime("%Y-%m-%d"),
    "through",
    BEFORE_END.strftime("%Y-%m-%d"),
)


print(
    "\nAfter:"
)


print(
    AFTER_START.strftime("%Y-%m-%d"),
    "through",
    AFTER_END.strftime("%Y-%m-%d"),
)


# =============================================================================
# 4. Create folder structure
# =============================================================================

for sensor_root in [

    S1_DAILY_DIR,

    S2_DAILY_DIR,

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (
                sensor_root /
                group /
                period
            )


            folder.mkdir(
                parents=True,
                exist_ok=True,
            )


print(
    "\nDaily dataset root:"
)


print(
    DAILY_DIR
)


# =============================================================================
# 5. Check required files
# =============================================================================

required_files = [

    TREATMENT_SITES_FILE,

    COUNTERFACTUAL_SITES_FILE,

    MATCHING_TABLE_FILE,

]


missing_files = [

    file_path

    for file_path
    in required_files

    if not file_path.exists()

]


if missing_files:

    raise FileNotFoundError(
        "Required input files are missing:\n\n"
        +
        "\n".join(
            str(
                file_path
            )
            for file_path
            in missing_files
        )
    )


# =============================================================================
# 6. Earth Engine
# =============================================================================

EARTH_ENGINE_PROJECT = (
    "hurricane-504721"
)


try:

    ee.Initialize(
        project=
            EARTH_ENGINE_PROJECT
    )


    print(
        "\nEarth Engine initialized."
    )


except Exception:

    print(
        "\nEarth Engine authentication required."
    )


    ee.Authenticate()


    ee.Initialize(
        project=
            EARTH_ENGINE_PROJECT
    )


# =============================================================================
# 7. Download settings
# =============================================================================

EXPORT_SCALE = 10


EXPORT_CRS = (
    "EPSG:32617"
)


MAX_RETRIES = 3


REQUEST_PAUSE_SECONDS = 0.20


# Existing valid TIFFs should not be downloaded again.
SKIP_EXISTING = True


# Sentinel-2 scene-level cloud threshold.
S2_MAX_CLOUD_PERCENT = 80


# =============================================================================
# 8. Current sample settings
# =============================================================================

N_TREATMENT_SITES = 10


N_CONTROLS_PER_TREATMENT = 5


# =============================================================================
# 9. Bands
# =============================================================================

S2_BANDS = [

    "B2",

    "B3",

    "B4",

    "B8",

    "B11",

    "B12",

    "NDVI",

    "NDWI",

]


S1_BANDS = [

    "VV",

    "VH",

    "VV_minus_VH",

]


# =============================================================================
# 10. Load sites
# =============================================================================

treatment_sites = gpd.read_file(
    TREATMENT_SITES_FILE
)


counterfactual_sites = gpd.read_file(
    COUNTERFACTUAL_SITES_FILE
)


matching_table = pd.read_csv(
    MATCHING_TABLE_FILE
)


if treatment_sites.crs is None:

    treatment_sites = (
        treatment_sites
        .set_crs(
            "EPSG:4326"
        )
    )


if counterfactual_sites.crs is None:

    counterfactual_sites = (
        counterfactual_sites
        .set_crs(
            "EPSG:4326"
        )
    )


treatment_sites = (
    treatment_sites
    .to_crs(
        "EPSG:4326"
    )
)


counterfactual_sites = (
    counterfactual_sites
    .to_crs(
        "EPSG:4326"
    )
)


if "site_id" not in treatment_sites.columns:

    raise ValueError(
        "treatment_sites.geojson does not contain site_id."
    )


if "site_id" not in counterfactual_sites.columns:

    raise ValueError(
        "counterfactual_sites.geojson does not contain site_id."
    )


treatment_sites[
    "site_id"
] = (
    treatment_sites[
        "site_id"
    ]
    .astype(str)
)


counterfactual_sites[
    "site_id"
] = (
    counterfactual_sites[
        "site_id"
    ]
    .astype(str)
)


print(
    "\nAll available treatment sites:",
    len(
        treatment_sites
    )
)


print(
    "All available counterfactual sites:",
    len(
        counterfactual_sites
    )
)


# =============================================================================
# 11. Identify matching-table columns
# =============================================================================

treatment_match_column = None


for candidate in [

    "treatment_site_id",

    "matched_treatment_site_id",

    "matched_treatment_id",

]:

    if candidate in matching_table.columns:

        treatment_match_column = (
            candidate
        )

        break


if treatment_match_column is None:

    raise ValueError(
        "Could not identify treatment ID column in site_matching_table.csv."
    )


counterfactual_match_column = None


for candidate in [

    "counterfactual_site_id",

    "site_id",

]:

    if candidate in matching_table.columns:

        counterfactual_match_column = (
            candidate
        )

        break


if counterfactual_match_column is None:

    raise ValueError(
        "Could not identify counterfactual ID column "
        "in site_matching_table.csv."
    )


matching_table[
    treatment_match_column
] = (
    matching_table[
        treatment_match_column
    ]
    .astype(str)
)


matching_table[
    counterfactual_match_column
] = (
    matching_table[
        counterfactual_match_column
    ]
    .astype(str)
)


# =============================================================================
# 12. Select first 10 treatment sites
# =============================================================================

treatment_to_process = (
    treatment_sites
    .iloc[
        :N_TREATMENT_SITES
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


selected_treatment_ids = (
    treatment_to_process[
        "site_id"
    ]
    .astype(str)
    .tolist()
)


print(
    "\nSelected treatment sites:"
)


for site_id in selected_treatment_ids:

    print(
        "  ",
        site_id
    )


# =============================================================================
# 13. Select first 5 matched controls for each treatment
# =============================================================================

selected_matching = (
    matching_table
    .loc[
        matching_table[
            treatment_match_column
        ]
        .isin(
            selected_treatment_ids
        )
    ]
    .copy()
)


# Prefer control_rank if it exists.
if "control_rank" in selected_matching.columns:

    selected_matching[
        "control_rank"
    ] = pd.to_numeric(
        selected_matching[
            "control_rank"
        ],
        errors=
            "coerce",
    )


    selected_matching = (
        selected_matching
        .sort_values(
            [
                treatment_match_column,
                "control_rank",
            ]
        )
        .copy()
    )


else:

    selected_matching = (
        selected_matching
        .sort_values(
            treatment_match_column
        )
        .copy()
    )


selected_matching = (
    selected_matching
    .groupby(
        treatment_match_column,
        group_keys=False,
    )
    .head(
        N_CONTROLS_PER_TREATMENT
    )
    .copy()
)


# =============================================================================
# 14. Validate 5 controls per treatment
# =============================================================================

control_counts = (
    selected_matching
    .groupby(
        treatment_match_column
    )[
        counterfactual_match_column
    ]
    .nunique()
    .reindex(
        selected_treatment_ids,
        fill_value=0,
    )
)


print(
    "\nControls selected per treatment:"
)


print(
    control_counts.to_string()
)


incomplete_treatments = (
    control_counts[
        control_counts
        <
        N_CONTROLS_PER_TREATMENT
    ]
)


if not incomplete_treatments.empty:

    print(
        "\nWARNING:"
    )


    print(
        "Some treatment sites have fewer than "
        f"{N_CONTROLS_PER_TREATMENT} controls."
    )


    print(
        incomplete_treatments.to_string()
    )


# =============================================================================
# 15. Counterfactual IDs
# =============================================================================

selected_counterfactual_ids = (
    selected_matching[
        counterfactual_match_column
    ]
    .astype(str)
    .drop_duplicates()
    .tolist()
)


counterfactual_to_process = (
    counterfactual_sites
    .loc[
        counterfactual_sites[
            "site_id"
        ]
        .isin(
            selected_counterfactual_ids
        )
    ]
    .copy()
)


# Preserve selected matching order.
control_order = {

    site_id:
        order

    for order, site_id
    in enumerate(
        selected_counterfactual_ids
    )

}


counterfactual_to_process[
    "_selection_order"
] = (
    counterfactual_to_process[
        "site_id"
    ]
    .map(
        control_order
    )
)


counterfactual_to_process = (
    counterfactual_to_process
    .sort_values(
        "_selection_order"
    )
    .drop(
        columns=
            "_selection_order"
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 16. Verify all selected controls exist
# =============================================================================

found_control_ids = set(
    counterfactual_to_process[
        "site_id"
    ]
)


missing_selected_controls = [

    site_id

    for site_id
    in selected_counterfactual_ids

    if site_id
    not in found_control_ids

]


if missing_selected_controls:

    raise ValueError(
        "Some selected counterfactual sites were not found in "
        "counterfactual_sites.geojson:\n\n"
        +
        "\n".join(
            missing_selected_controls
        )
    )


# =============================================================================
# 17. Save selected sample
# =============================================================================

treatment_sample_table = pd.DataFrame(
    {

        "site_id":
            treatment_to_process[
                "site_id"
            ],

        "group":
            "treatment",

        "matched_treatment_site_id":
            treatment_to_process[
                "site_id"
            ],

        "control_rank":
            np.nan,

    }
)


control_sample_table = (
    selected_matching[
        [
            treatment_match_column,
            counterfactual_match_column,
        ]
        +
        (
            [
                "control_rank"
            ]
            if "control_rank"
            in selected_matching.columns
            else []
        )
    ]
    .copy()
)


control_sample_table = (
    control_sample_table
    .rename(
        columns={

            treatment_match_column:
                "matched_treatment_site_id",

            counterfactual_match_column:
                "site_id",

        }
    )
)


control_sample_table[
    "group"
] = (
    "counterfactual"
)


if "control_rank" not in control_sample_table.columns:

    control_sample_table[
        "control_rank"
    ] = (
        control_sample_table
        .groupby(
            "matched_treatment_site_id"
        )
        .cumcount()
        +
        1
    )


selected_sample = pd.concat(
    [
        treatment_sample_table,
        control_sample_table[
            [
                "site_id",
                "group",
                "matched_treatment_site_id",
                "control_rank",
            ]
        ],
    ],
    ignore_index=True,
)


SELECTED_SAMPLE_FILE = (
    DAILY_DIR /
    "selected_site_sample.csv"
)


selected_sample.to_csv(
    SELECTED_SAMPLE_FILE,
    index=False,
)


print(
    "\n"
    + "=" * 90
)


print(
    "CURRENT SAMPLE"
)


print(
    "=" * 90
)


print(
    "\nTreatment sites:",
    len(
        treatment_to_process
    )
)


print(
    "Counterfactual sites:",
    len(
        counterfactual_to_process
    )
)


print(
    "Total sites:",
    (
        len(
            treatment_to_process
        )
        +
        len(
            counterfactual_to_process
        )
    )
)


print(
    "\nSelected sample saved:"
)


print(
    SELECTED_SAMPLE_FILE
)


# =============================================================================
# 18. Matching information helper
# =============================================================================

def get_matching_information(
    group,
    site_id,
):

    if group == "treatment":

        return {

            "matched_treatment_site_id":
                site_id,

            "control_rank":
                None,

        }


    selected = (
        selected_matching
        .loc[
            selected_matching[
                counterfactual_match_column
            ]
            ==
            str(
                site_id
            )
        ]
    )


    if selected.empty:

        return {

            "matched_treatment_site_id":
                None,

            "control_rank":
                None,

        }


    row = (
        selected.iloc[0]
    )


    return {

        "matched_treatment_site_id":
            str(
                row[
                    treatment_match_column
                ]
            ),

        "control_rank":
            (
                int(
                    row[
                        "control_rank"
                    ]
                )
                if (
                    "control_rank"
                    in selected.columns
                    and
                    pd.notna(
                        row[
                            "control_rank"
                        ]
                    )
                )
                else None
            ),

    }


# =============================================================================
# 19. Site geometry -> Earth Engine
# =============================================================================

def site_geometry_to_ee(
    row,
):

    geometry = (
        row.geometry
    )


    if geometry.geom_type in [

        "Point",

        "MultiPoint",

    ]:

        center = (
            geometry.centroid
        )


        return (
            ee.Geometry.Point(
                [
                    float(
                        center.x
                    ),
                    float(
                        center.y
                    ),
                ]
            )
            .buffer(
                500
            )
            .bounds()
        )


    return ee.Geometry(
        geometry.__geo_interface__
    )


# =============================================================================
# 20. Sentinel-2 cloud mask
# =============================================================================

def mask_sentinel2(
    image,
):

    scl = (
        image
        .select(
            "SCL"
        )
    )


    clear = (
        scl.neq(3)
        .And(
            scl.neq(8)
        )
        .And(
            scl.neq(9)
        )
        .And(
            scl.neq(10)
        )
        .And(
            scl.neq(11)
        )
    )


    return (
        image
        .updateMask(
            clear
        )
    )


# =============================================================================
# 21. Prepare Sentinel-2
# =============================================================================

def prepare_sentinel2_image(
    image,
):

    masked = (
        mask_sentinel2(
            image
        )
    )


    reflectance = (
        masked
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B8",
                "B11",
                "B12",
            ]
        )
        .multiply(
            0.0001
        )
        .rename(
            [
                "B2",
                "B3",
                "B4",
                "B8",
                "B11",
                "B12",
            ]
        )
    )


    ndvi = (
        reflectance
        .normalizedDifference(
            [
                "B8",
                "B4",
            ]
        )
        .rename(
            "NDVI"
        )
    )


    ndwi = (
        reflectance
        .normalizedDifference(
            [
                "B3",
                "B8",
            ]
        )
        .rename(
            "NDWI"
        )
    )


    return (
        reflectance
        .addBands(
            ndvi
        )
        .addBands(
            ndwi
        )
        .toFloat()
        .copyProperties(
            image,
            image.propertyNames()
        )
    )


# =============================================================================
# 22. Sentinel-2 collection
# =============================================================================

def get_sentinel2_collection(
    geometry,
):

    return (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(
            geometry
        )
        .filterDate(
            STUDY_START.strftime(
                "%Y-%m-%d"
            ),
            EE_STUDY_END,
        )
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                S2_MAX_CLOUD_PERCENT,
            )
        )
        .map(
            prepare_sentinel2_image
        )
    )


# =============================================================================
# 23. Prepare Sentinel-1
# =============================================================================

def prepare_sentinel1_image(
    image,
):

    vv = (
        image
        .select(
            "VV"
        )
        .rename(
            "VV"
        )
    )


    vh = (
        image
        .select(
            "VH"
        )
        .rename(
            "VH"
        )
    )


    vv_minus_vh = (
        vv
        .subtract(
            vh
        )
        .rename(
            "VV_minus_VH"
        )
    )


    return (
        vv
        .addBands(
            vh
        )
        .addBands(
            vv_minus_vh
        )
        .toFloat()
        .copyProperties(
            image,
            image.propertyNames()
        )
    )


# =============================================================================
# 24. Sentinel-1 collection
# =============================================================================

def get_sentinel1_collection(
    geometry,
):

    return (
        ee.ImageCollection(
            "COPERNICUS/S1_GRD"
        )
        .filterBounds(
            geometry
        )
        .filterDate(
            STUDY_START.strftime(
                "%Y-%m-%d"
            ),
            EE_STUDY_END,
        )
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH",
            )
        )
        .filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                "ASCENDING",
            )
        )
        .map(
            prepare_sentinel1_image
        )
    )


# =============================================================================
# 25. Acquisition dates
# =============================================================================

def get_collection_dates(
    collection,
):

    timestamps = (
        collection
        .aggregate_array(
            "system:time_start"
        )
        .getInfo()
    )


    return sorted(
        {

            datetime
            .fromtimestamp(
                timestamp / 1000,
                tz=timezone.utc,
            )
            .strftime(
                "%Y-%m-%d"
            )

            for timestamp
            in timestamps

        }
    )


# =============================================================================
# 26. TIFF inspection
# =============================================================================

def inspect_downloaded_geotiff(
    file_path,
    band_names,
):

    file_path = Path(
        file_path
    )


    if not file_path.exists():

        return {

            "reusable":
                False,

            "readable":
                False,

            "validation_status":
                "missing",

            "error":
                "File does not exist.",

        }


    try:

        with rasterio.open(
            file_path
        ) as src:

            data = (
                src
                .read(
                    masked=True
                )
                .astype(
                    "float32"
                )
                .filled(
                    np.nan
                )
            )


            band_count = int(
                src.count
            )


            width = int(
                src.width
            )


            height = int(
                src.height
            )


            crs = (
                str(
                    src.crs
                )
                if src.crs is not None
                else None
            )


            resolution_x = float(
                src.res[0]
            )


            resolution_y = float(
                src.res[1]
            )


        if band_count != len(
            band_names
        ):

            return {

                "reusable":
                    False,

                "readable":
                    True,

                "validation_status":
                    "wrong_band_count",

                "error":
                    (
                        f"Expected {len(band_names)} bands, "
                        f"found {band_count}."
                    ),

            }


        if (
            width <= 0
            or
            height <= 0
        ):

            return {

                "reusable":
                    False,

                "readable":
                    True,

                "validation_status":
                    "invalid_dimensions",

                "error":
                    "Invalid raster dimensions.",

            }


        if crs is None:

            return {

                "reusable":
                    False,

                "readable":
                    True,

                "validation_status":
                    "missing_crs",

                "error":
                    "Raster CRS missing.",

            }


        finite = np.isfinite(
            data
        )


        # PRIMARY
        valid_any = (
            finite
            .any(
                axis=0
            )
        )


        # STRICT DIAGNOSTIC
        valid_all = (
            finite
            .all(
                axis=0
            )
        )


        result = {

            "reusable":
                True,

            "readable":
                True,

            "validation_status":
                "valid",

            "error":
                None,

            "band_count":
                band_count,

            "width":
                width,

            "height":
                height,

            "crs":
                crs,

            "resolution_x":
                resolution_x,

            "resolution_y":
                resolution_y,

            "valid_pixel_fraction":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            "valid_pixel_fraction_any_band":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage_any_band":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            "valid_pixel_fraction_all_bands":
                float(
                    valid_all.mean()
                ),

            "valid_pixel_percentage_all_bands":
                float(
                    valid_all.mean()
                    *
                    100
                ),

        }


        for band_index, band_name in enumerate(
            band_names
        ):

            fraction = float(
                finite[
                    band_index
                ].mean()
            )


            result[
                f"valid_fraction_{band_name}"
            ] = (
                fraction
            )


            result[
                f"valid_percentage_{band_name}"
            ] = (
                fraction
                *
                100
            )


        return result


    except Exception as error:

        return {

            "reusable":
                False,

            "readable":
                False,

            "validation_status":
                "corrupt_or_unreadable",

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 27. Existing inventory metadata
# =============================================================================

OLD_INVENTORY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


if OLD_INVENTORY_FILE.exists():

    try:

        old_inventory = pd.read_csv(
            OLD_INVENTORY_FILE
        )


        old_inventory[
            "site_id"
        ] = (
            old_inventory[
                "site_id"
            ]
            .astype(str)
        )


        old_inventory[
            "acquisition_date"
        ] = (
            old_inventory[
                "acquisition_date"
            ]
            .astype(str)
        )


        print(
            "\nExisting inventory rows available for metadata reuse:",
            len(
                old_inventory
            )
        )


    except Exception:

        old_inventory = pd.DataFrame()


else:

    old_inventory = pd.DataFrame()


def get_old_inventory_metadata(
    site_id,
    group,
    sensor,
    acquisition_date,
):

    if old_inventory.empty:

        return {}


    selected = (
        old_inventory
        .loc[
            (
                old_inventory[
                    "site_id"
                ].astype(str)
                ==
                str(
                    site_id
                )
            )
            &
            (
                old_inventory[
                    "group"
                ].astype(str)
                ==
                str(
                    group
                )
            )
            &
            (
                old_inventory[
                    "sensor"
                ].astype(str)
                ==
                str(
                    sensor
                )
            )
            &
            (
                old_inventory[
                    "acquisition_date"
                ].astype(str)
                ==
                str(
                    acquisition_date
                )
            )
        ]
    )


    if selected.empty:

        return {}


    row = (
        selected.iloc[0]
    )


    reusable_columns = [

        "same_day_scene_count",

        "acquisition_time_start",

        "acquisition_time_end",

        "source_image_ids",

        "cloudy_pixel_percentage_mean",

        "cloudy_pixel_percentage_max",

        "mgrs_tiles",

        "sensing_orbit_numbers",

        "orbit_pass",

        "platform_numbers",

        "orbit_numbers",

        "relative_orbit_numbers",

    ]


    result = {}


    for column in reusable_columns:

        if (
            column in selected.columns
            and
            pd.notna(
                row[
                    column
                ]
            )
        ):

            result[
                column
            ] = (
                row[
                    column
                ]
            )


    return result


# =============================================================================
# 28. Metadata utilities
# =============================================================================

def timestamp_ms_to_iso(
    timestamp,
):

    if timestamp is None:

        return None


    return (
        datetime
        .fromtimestamp(
            timestamp / 1000,
            tz=timezone.utc,
        )
        .isoformat()
    )


def clean_unique_list(
    values,
):

    result = []


    for value in values:

        if value is None:

            continue


        value = str(
            value
        )


        if value not in result:

            result.append(
                value
            )


    return result


# =============================================================================
# 29. Daily EE metadata
# =============================================================================

def get_daily_metadata(
    daily_collection,
    sensor,
):

    scene_count = int(
        daily_collection
        .size()
        .getInfo()
    )


    timestamps = (
        daily_collection
        .aggregate_array(
            "system:time_start"
        )
        .getInfo()
    )


    timestamp_strings = sorted(
        clean_unique_list(
            [
                timestamp_ms_to_iso(
                    timestamp
                )
                for timestamp
                in timestamps
            ]
        )
    )


    source_ids = (
        daily_collection
        .aggregate_array(
            "system:index"
        )
        .getInfo()
    )


    metadata = {

        "same_day_scene_count":
            scene_count,

        "acquisition_time_start":
            (
                timestamp_strings[0]
                if timestamp_strings
                else None
            ),

        "acquisition_time_end":
            (
                timestamp_strings[-1]
                if timestamp_strings
                else None
            ),

        "source_image_ids":
            json.dumps(
                clean_unique_list(
                    source_ids
                )
            ),

        "cloudy_pixel_percentage_mean":
            None,

        "cloudy_pixel_percentage_max":
            None,

        "mgrs_tiles":
            None,

        "sensing_orbit_numbers":
            None,

        "orbit_pass":
            None,

        "platform_numbers":
            None,

        "orbit_numbers":
            None,

        "relative_orbit_numbers":
            None,

    }


    if sensor == "sentinel2":

        cloud_values = (
            daily_collection
            .aggregate_array(
                "CLOUDY_PIXEL_PERCENTAGE"
            )
            .getInfo()
        )


        cloud_values = [

            float(
                value
            )

            for value
            in cloud_values

            if value is not None

        ]


        if cloud_values:

            metadata[
                "cloudy_pixel_percentage_mean"
            ] = float(
                np.mean(
                    cloud_values
                )
            )


            metadata[
                "cloudy_pixel_percentage_max"
            ] = float(
                np.max(
                    cloud_values
                )
            )


        metadata[
            "mgrs_tiles"
        ] = json.dumps(
            clean_unique_list(
                daily_collection
                .aggregate_array(
                    "MGRS_TILE"
                )
                .getInfo()
            )
        )


        metadata[
            "sensing_orbit_numbers"
        ] = json.dumps(
            clean_unique_list(
                daily_collection
                .aggregate_array(
                    "SENSING_ORBIT_NUMBER"
                )
                .getInfo()
            )
        )


    elif sensor == "sentinel1":

        metadata[
            "orbit_pass"
        ] = ",".join(
            clean_unique_list(
                daily_collection
                .aggregate_array(
                    "orbitProperties_pass"
                )
                .getInfo()
            )
        )


        metadata[
            "platform_numbers"
        ] = json.dumps(
            clean_unique_list(
                daily_collection
                .aggregate_array(
                    "platform_number"
                )
                .getInfo()
            )
        )


        metadata[
            "orbit_numbers"
        ] = json.dumps(
            clean_unique_list(
                daily_collection
                .aggregate_array(
                    "orbitNumber_start"
                )
                .getInfo()
            )
        )


        metadata[
            "relative_orbit_numbers"
        ] = json.dumps(
            clean_unique_list(
                daily_collection
                .aggregate_array(
                    "relativeOrbitNumber_start"
                )
                .getInfo()
            )
        )


    return metadata


# =============================================================================
# 30. Build same-day Earth Engine image
# =============================================================================

def build_acquisition_image(
    collection,
    acquisition_date,
    geometry,
    sensor,
):

    day_start = (
        ee.Date(
            acquisition_date
        )
    )


    day_end = (
        day_start
        .advance(
            1,
            "day"
        )
    )


    daily_collection = (
        collection
        .filterDate(
            day_start,
            day_end,
        )
    )


    metadata = (
        get_daily_metadata(
            daily_collection,
            sensor,
        )
    )


    image = (
        daily_collection
        .mosaic()
        .clip(
            geometry
        )
    )


    return (
        image,
        metadata,
    )


# =============================================================================
# 31. Invalid file backup
# =============================================================================

def get_invalid_backup_path(
    output_file,
):

    output_file = Path(
        output_file
    )


    candidate = (
        output_file
        .with_name(
            output_file.name
            +
            ".invalid"
        )
    )


    counter = 1


    while candidate.exists():

        candidate = (
            output_file
            .with_name(
                output_file.name
                +
                f".invalid_{counter}"
            )
        )


        counter += 1


    return candidate


# =============================================================================
# 32. Download new image
# =============================================================================

def download_new_image(
    image,
    region,
    output_file,
    band_names,
):

    output_file = Path(
        output_file
    )


    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    last_error = None


    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            print(
                f"    Download attempt {attempt}..."
            )


            geemap.download_ee_image(

                image,

                filename=str(
                    output_file
                ),

                region=
                    region,

                scale=
                    EXPORT_SCALE,

                crs=
                    EXPORT_CRS,

            )


            inspection = (
                inspect_downloaded_geotiff(
                    output_file,
                    band_names,
                )
            )


            if inspection.get(
                "reusable",
                False
            ):

                inspection[
                    "status"
                ] = (
                    "success"
                )


                inspection[
                    "download_action"
                ] = (
                    "downloaded"
                )


                inspection[
                    "file_size_bytes"
                ] = (
                    output_file
                    .stat()
                    .st_size
                )


                return inspection


            last_error = (
                inspection.get(
                    "error"
                )
            )


        except Exception as error:

            last_error = str(
                error
            )


        print(
            "    Download failed:",
            last_error
        )


        if output_file.exists():

            try:

                output_file.unlink()

            except Exception:

                pass


        if attempt < MAX_RETRIES:

            time.sleep(
                2
            )


    return {

        "status":
            "failed",

        "download_action":
            "failed",

        "reusable":
            False,

        "readable":
            False,

        "validation_status":
            "download_failed",

        "error":
            last_error,

        "file_size_bytes":
            np.nan,

        "valid_pixel_fraction":
            np.nan,

        "valid_pixel_percentage":
            np.nan,

        "valid_pixel_fraction_any_band":
            np.nan,

        "valid_pixel_percentage_any_band":
            np.nan,

        "valid_pixel_fraction_all_bands":
            np.nan,

        "valid_pixel_percentage_all_bands":
            np.nan,

    }


# =============================================================================
# 33. Process one site — LOCAL FIRST
# =============================================================================

def process_site(
    sensor,
    group,
    site_id,
    geometry,
):

    print(
        "\n"
        + "=" * 90
    )


    print(
        sensor.upper(),
        "|",
        group.upper(),
        "|",
        site_id,
    )


    print(
        "=" * 90
    )


    # -------------------------------------------------------------------------
    # Sensor configuration
    # -------------------------------------------------------------------------

    if sensor == "sentinel2":

        collection = (
            get_sentinel2_collection(
                geometry
            )
        )


        sensor_root = (
            S2_DAILY_DIR
        )


        band_names = (
            S2_BANDS
        )


    elif sensor == "sentinel1":

        collection = (
            get_sentinel1_collection(
                geometry
            )
        )


        sensor_root = (
            S1_DAILY_DIR
        )


        band_names = (
            S1_BANDS
        )


    else:

        raise ValueError(
            f"Unknown sensor: {sensor}"
        )


    # -------------------------------------------------------------------------
    # One EE request to identify acquisition dates.
    # -------------------------------------------------------------------------

    acquisition_dates = (
        get_collection_dates(
            collection
        )
    )


    print(
        "Unique acquisition dates:",
        len(
            acquisition_dates
        )
    )


    matching_information = (
        get_matching_information(
            group,
            site_id,
        )
    )


    records = []


    # =========================================================================
    # Acquisition-date loop
    # =========================================================================

    for acquisition_number, acquisition_date in enumerate(
        acquisition_dates,
        start=1,
    ):

        period = (
            assign_period(
                acquisition_date
            )
        )


        days_relative_to_helene = (
            pd.Timestamp(
                acquisition_date
            )
            -
            HELENE_REFERENCE_DATE
        ).days


        print(
            f"\n  {acquisition_number}/"
            f"{len(acquisition_dates)} "
            f"{acquisition_date} "
            f"[{period}]"
        )


        # ---------------------------------------------------------------------
        # Expected local path
        # -------------------------------------------------------------------------

        output_file = (

            sensor_root /
            group /
            period /
            (
                f"{site_id}_"
                f"{acquisition_date}_"
                f"{sensor}.tif"
            )

        )


        # =====================================================================
        # LOCAL FILE CHECK FIRST
        # =====================================================================

        if (
            SKIP_EXISTING
            and
            output_file.exists()
        ):

            inspection = (
                inspect_downloaded_geotiff(
                    output_file,
                    band_names,
                )
            )


            # -----------------------------------------------------------------
            # Existing valid TIFF.
            # -----------------------------------------------------------------

            if inspection.get(
                "reusable",
                False
            ):

                print(
                    "    Existing valid TIFF found — "
                    "skipping EE image build and download."
                )


                inspection[
                    "status"
                ] = (
                    "existing"
                )


                inspection[
                    "download_action"
                ] = (
                    "skipped_existing"
                )


                inspection[
                    "file_size_bytes"
                ] = (
                    output_file
                    .stat()
                    .st_size
                )


                ee_metadata = (
                    get_old_inventory_metadata(

                        site_id=
                            site_id,

                        group=
                            group,

                        sensor=
                            sensor,

                        acquisition_date=
                            acquisition_date,

                    )
                )


                record = {

                    "site_id":
                        site_id,

                    "group":
                        group,

                    "matched_treatment_site_id":
                        matching_information[
                            "matched_treatment_site_id"
                        ],

                    "control_rank":
                        matching_information[
                            "control_rank"
                        ],

                    "sensor":
                        sensor,

                    "acquisition_date":
                        acquisition_date,

                    "period":
                        period,

                    "helene_reference_date":
                        "2024-09-27",

                    "days_relative_to_helene":
                        days_relative_to_helene,

                    "file_path":
                        str(
                            output_file
                        ),

                }


                record.update(
                    ee_metadata
                )


                record.update(
                    inspection
                )


                records.append(
                    record
                )


                continue


            # -----------------------------------------------------------------
            # Existing TIFF is invalid.
            # -----------------------------------------------------------------

            print(
                "    Existing TIFF failed validation:"
            )


            print(
                "   ",
                inspection.get(
                    "validation_status"
                ),
                "|",
                inspection.get(
                    "error"
                ),
            )


            invalid_backup = (
                get_invalid_backup_path(
                    output_file
                )
            )


            try:

                output_file.rename(
                    invalid_backup
                )


                print(
                    "    Invalid file moved to:"
                )


                print(
                    "   ",
                    invalid_backup
                )


            except Exception:

                try:

                    output_file.unlink()

                except Exception:

                    pass


        # =====================================================================
        # Local TIFF unavailable -> Earth Engine
        # =====================================================================

        print(
            "    Local TIFF not available — querying Earth Engine."
        )


        try:

            acquisition_image, ee_metadata = (
                build_acquisition_image(

                    collection=
                        collection,

                    acquisition_date=
                        acquisition_date,

                    geometry=
                        geometry,

                    sensor=
                        sensor,

                )
            )


            local_metadata = (
                download_new_image(

                    image=
                        acquisition_image,

                    region=
                        geometry,

                    output_file=
                        output_file,

                    band_names=
                        band_names,

                )
            )


        except Exception as error:

            print(
                "    Earth Engine/image construction failed:"
            )


            print(
                "   ",
                error
            )


            ee_metadata = {}


            local_metadata = {

                "status":
                    "failed",

                "download_action":
                    "failed",

                "reusable":
                    False,

                "readable":
                    False,

                "validation_status":
                    "ee_query_failed",

                "error":
                    str(
                        error
                    ),

                "valid_pixel_fraction":
                    np.nan,

                "valid_pixel_percentage":
                    np.nan,

            }


        record = {

            "site_id":
                site_id,

            "group":
                group,

            "matched_treatment_site_id":
                matching_information[
                    "matched_treatment_site_id"
                ],

            "control_rank":
                matching_information[
                    "control_rank"
                ],

            "sensor":
                sensor,

            "acquisition_date":
                acquisition_date,

            "period":
                period,

            "helene_reference_date":
                "2024-09-27",

            "days_relative_to_helene":
                days_relative_to_helene,

            "file_path":
                str(
                    output_file
                ),

        }


        record.update(
            ee_metadata
        )


        record.update(
            local_metadata
        )


        records.append(
            record
        )


        time.sleep(
            REQUEST_PAUSE_SECONDS
        )


    return records


# =============================================================================
# 34. Process group
# =============================================================================

def process_group(
    sensor,
    group,
    sites,
):

    records = []


    total_sites = len(
        sites
    )


    for site_number, (_, row) in enumerate(
        sites.iterrows(),
        start=1,
    ):

        site_id = str(
            row[
                "site_id"
            ]
        )


        print(
            "\n\n"
            + "#" * 100
        )


        print(
            f"{sensor.upper()} | "
            f"{group.upper()} | "
            f"SITE {site_number}/{total_sites}"
        )


        print(
            site_id
        )


        print(
            "#" * 100
        )


        geometry = (
            site_geometry_to_ee(
                row
            )
        )


        try:

            site_records = (
                process_site(

                    sensor=
                        sensor,

                    group=
                        group,

                    site_id=
                        site_id,

                    geometry=
                        geometry,

                )
            )


            records.extend(
                site_records
            )


        except Exception as error:

            print(
                "\nERROR processing site:"
            )


            print(
                site_id
            )


            print(
                error
            )


            print(
                "Continuing to next site."
            )


    return records


# =============================================================================
# 35. Process Sentinel-2 treatment
# =============================================================================

s2_treatment_records = (
    process_group(

        sensor=
            "sentinel2",

        group=
            "treatment",

        sites=
            treatment_to_process,

    )
)


# =============================================================================
# 36. Process Sentinel-2 counterfactual
# =============================================================================

s2_counterfactual_records = (
    process_group(

        sensor=
            "sentinel2",

        group=
            "counterfactual",

        sites=
            counterfactual_to_process,

    )
)


# =============================================================================
# 37. Save Sentinel-2 inventory immediately
# =============================================================================

s2_inventory = pd.DataFrame(
    s2_treatment_records
    +
    s2_counterfactual_records
)


S2_INVENTORY_FILE = (
    DAILY_DIR /
    "sentinel2_daily_inventory.csv"
)


s2_inventory.to_csv(
    S2_INVENTORY_FILE,
    index=False,
)


print(
    "\nSentinel-2 inventory saved:"
)


print(
    S2_INVENTORY_FILE
)


# =============================================================================
# 38. Process Sentinel-1 treatment
# =============================================================================

s1_treatment_records = (
    process_group(

        sensor=
            "sentinel1",

        group=
            "treatment",

        sites=
            treatment_to_process,

    )
)


# =============================================================================
# 39. Process Sentinel-1 counterfactual
# =============================================================================

s1_counterfactual_records = (
    process_group(

        sensor=
            "sentinel1",

        group=
            "counterfactual",

        sites=
            counterfactual_to_process,

    )
)


# =============================================================================
# 40. Save Sentinel-1 inventory
# =============================================================================

s1_inventory = pd.DataFrame(
    s1_treatment_records
    +
    s1_counterfactual_records
)


S1_INVENTORY_FILE = (
    DAILY_DIR /
    "sentinel1_daily_inventory.csv"
)


s1_inventory.to_csv(
    S1_INVENTORY_FILE,
    index=False,
)


print(
    "\nSentinel-1 inventory saved:"
)


print(
    S1_INVENTORY_FILE
)


# =============================================================================
# 41. Combined inventory
# =============================================================================

combined_inventory = pd.concat(
    [
        s1_inventory,
        s2_inventory,
    ],
    ignore_index=True,
)


COMBINED_INVENTORY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


combined_inventory.to_csv(
    COMBINED_INVENTORY_FILE,
    index=False,
)


print(
    "\nCombined inventory saved:"
)


print(
    COMBINED_INVENTORY_FILE
)


# =============================================================================
# 42. Download/reuse summary
# =============================================================================

if (
    not combined_inventory.empty
    and
    "download_action"
    in combined_inventory.columns
):

    download_summary = (
        combined_inventory[
            "download_action"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "download_action"
        )
        .reset_index(
            name=
                "image_count"
        )
    )


    DOWNLOAD_SUMMARY_FILE = (
        DAILY_DIR /
        "daily_download_action_summary.csv"
    )


    download_summary.to_csv(
        DOWNLOAD_SUMMARY_FILE,
        index=False,
    )


    print(
        "\n"
        + "=" * 90
    )


    print(
        "DOWNLOAD / REUSE SUMMARY"
    )


    print(
        "=" * 90
    )


    print(
        download_summary.to_string(
            index=False
        )
    )


# =============================================================================
# 43. Complete calendar
# =============================================================================

calendar_dates = pd.DataFrame(
    {

        "date":
            pd.date_range(

                start=
                    STUDY_START,

                end=
                    STUDY_END,

                freq=
                    "D",

            )

    }
)


calendar_dates[
    "period"
] = (
    calendar_dates[
        "date"
    ]
    .apply(
        assign_period
    )
)


calendar_dates[
    "helene_reference_date"
] = (
    HELENE_REFERENCE_DATE
)


calendar_dates[
    "days_relative_to_helene"
] = (
    calendar_dates[
        "date"
    ]
    -
    HELENE_REFERENCE_DATE
).dt.days


print(
    "\nCalendar days:",
    len(
        calendar_dates
    )
)


# =============================================================================
# 44. Site master
# =============================================================================

site_master = (
    selected_sample.copy()
)


site_master[
    "site_id"
] = (
    site_master[
        "site_id"
    ]
    .astype(str)
)


# =============================================================================
# 45. Cross join site × calendar
# =============================================================================

site_master[
    "_calendar_key"
] = 1


calendar_dates[
    "_calendar_key"
] = 1


daily_calendar = (
    site_master
    .merge(
        calendar_dates,
        on=
            "_calendar_key",
    )
    .drop(
        columns=
            "_calendar_key"
    )
)


# =============================================================================
# 46. Prepare sensor inventory for calendar
# =============================================================================

def prepare_sensor_calendar_data(
    sensor_inventory,
    prefix,
):

    if sensor_inventory.empty:

        return pd.DataFrame(
            columns=[
                "site_id",
                "group",
                "date",
            ]
        )


    table = (
        sensor_inventory.copy()
    )


    table[
        "date"
    ] = pd.to_datetime(
        table[
            "acquisition_date"
        ]
    )


    possible_columns = [

        "site_id",

        "group",

        "date",

        "file_path",

        "status",

        "download_action",

        "validation_status",

        "valid_pixel_fraction",

        "valid_pixel_percentage",

        "valid_pixel_fraction_any_band",

        "valid_pixel_percentage_any_band",

        "valid_pixel_fraction_all_bands",

        "valid_pixel_percentage_all_bands",

        "same_day_scene_count",

        "acquisition_time_start",

        "acquisition_time_end",

        "cloudy_pixel_percentage_mean",

        "cloudy_pixel_percentage_max",

        "relative_orbit_numbers",

        "orbit_pass",

    ]


    keep_columns = [

        column

        for column
        in possible_columns

        if column in table.columns

    ]


    table = (
        table[
            keep_columns
        ]
        .copy()
    )


    rename_dict = {

        column:
            f"{prefix}_{column}"

        for column
        in table.columns

        if column not in [

            "site_id",

            "group",

            "date",

        ]

    }


    return (
        table
        .rename(
            columns=
                rename_dict
        )
    )


# =============================================================================
# 47. Merge S1/S2 onto calendar
# =============================================================================

s1_calendar_data = (
    prepare_sensor_calendar_data(
        s1_inventory,
        "sentinel1",
    )
)


s2_calendar_data = (
    prepare_sensor_calendar_data(
        s2_inventory,
        "sentinel2",
    )
)


daily_calendar = (
    daily_calendar
    .merge(
        s1_calendar_data,
        on=[
            "site_id",
            "group",
            "date",
        ],
        how=
            "left",
    )
    .merge(
        s2_calendar_data,
        on=[
            "site_id",
            "group",
            "date",
        ],
        how=
            "left",
    )
)


# =============================================================================
# 48. Availability indicators
# =============================================================================

for sensor in [

    "sentinel1",

    "sentinel2",

]:

    file_column = (
        f"{sensor}_file_path"
    )


    if file_column not in daily_calendar.columns:

        daily_calendar[
            file_column
        ] = (
            np.nan
        )


    daily_calendar[
        f"{sensor}_available"
    ] = (
        daily_calendar[
            file_column
        ]
        .notna()
        .astype(int)
    )


    daily_calendar[
        f"{sensor}_missing"
    ] = (
        1
        -
        daily_calendar[
            f"{sensor}_available"
        ]
    )


daily_calendar[
    "any_satellite_available"
] = (
    (
        daily_calendar[
            "sentinel1_available"
        ]
        +
        daily_calendar[
            "sentinel2_available"
        ]
    )
    >
    0
).astype(int)


daily_calendar[
    "both_satellites_available"
] = (
    (
        daily_calendar[
            "sentinel1_available"
        ]
        ==
        1
    )
    &
    (
        daily_calendar[
            "sentinel2_available"
        ]
        ==
        1
    )
).astype(int)


# =============================================================================
# 49. Save calendar
# =============================================================================

daily_calendar = (
    daily_calendar
    .sort_values(
        [
            "group",
            "site_id",
            "date",
        ]
    )
    .reset_index(
        drop=True
    )
)


DAILY_CALENDAR_FILE = (
    DAILY_DIR /
    "daily_calendar_index.csv"
)


daily_calendar.to_csv(
    DAILY_CALENDAR_FILE,
    index=False,
)


# =============================================================================
# 50. Dataset summary
# =============================================================================

summary_records = []


for sensor in [

    "sentinel1",

    "sentinel2",

]:

    availability_column = (
        f"{sensor}_available"
    )


    quality_column = (
        f"{sensor}_valid_pixel_fraction"
    )


    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            subset = (
                daily_calendar
                .loc[
                    (
                        daily_calendar[
                            "group"
                        ]
                        ==
                        group
                    )
                    &
                    (
                        daily_calendar[
                            "period"
                        ]
                        ==
                        period
                    )
                ]
                .copy()
            )


            if subset.empty:

                continue


            if quality_column in subset.columns:

                quality_values = (
                    subset[
                        quality_column
                    ]
                    .dropna()
                )


            else:

                quality_values = pd.Series(
                    dtype=float
                )


            observed_site_days = int(
                subset[
                    availability_column
                ].sum()
            )


            summary_records.append(
                {

                    "sensor":
                        sensor,

                    "group":
                        group,

                    "period":
                        period,

                    "number_of_sites":
                        subset[
                            "site_id"
                        ].nunique(),

                    "calendar_days_per_site":
                        subset[
                            "date"
                        ].nunique(),

                    "calendar_site_days":
                        len(
                            subset
                        ),

                    "observed_site_days":
                        observed_site_days,

                    "missing_site_days":
                        (
                            len(
                                subset
                            )
                            -
                            observed_site_days
                        ),

                    "observation_rate":
                        (
                            observed_site_days
                            /
                            len(
                                subset
                            )
                            if len(
                                subset
                            )
                            >
                            0
                            else np.nan
                        ),

                    "average_acquisitions_per_site":
                        float(
                            subset
                            .groupby(
                                "site_id"
                            )[
                                availability_column
                            ]
                            .sum()
                            .mean()
                        ),

                    "mean_valid_pixel_fraction":
                        (
                            float(
                                quality_values.mean()
                            )
                            if len(
                                quality_values
                            )
                            >
                            0
                            else np.nan
                        ),

                    "median_valid_pixel_fraction":
                        (
                            float(
                                quality_values.median()
                            )
                            if len(
                                quality_values
                            )
                            >
                            0
                            else np.nan
                        ),

                    "images_ge_80pct_valid":
                        (
                            int(
                                (
                                    quality_values
                                    >=
                                    0.80
                                )
                                .sum()
                            )
                            if len(
                                quality_values
                            )
                            >
                            0
                            else 0
                        ),

                    "percent_images_ge_80pct_valid":
                        (
                            float(
                                (
                                    quality_values
                                    >=
                                    0.80
                                )
                                .mean()
                                *
                                100
                            )
                            if len(
                                quality_values
                            )
                            >
                            0
                            else np.nan
                        ),

                }
            )


daily_summary = pd.DataFrame(
    summary_records
)


if not daily_summary.empty:

    daily_summary[
        "mean_valid_pixel_percentage"
    ] = (
        daily_summary[
            "mean_valid_pixel_fraction"
        ]
        *
        100
    )


    daily_summary[
        "median_valid_pixel_percentage"
    ] = (
        daily_summary[
            "median_valid_pixel_fraction"
        ]
        *
        100
    )


DAILY_SUMMARY_FILE = (
    DAILY_DIR /
    "daily_dataset_summary.csv"
)


daily_summary.to_csv(
    DAILY_SUMMARY_FILE,
    index=False,
)


# =============================================================================
# 51. Folder summary
# =============================================================================

folder_records = []


for sensor_name, sensor_root in [

    (
        "sentinel1",
        S1_DAILY_DIR,
    ),

    (
        "sentinel2",
        S2_DAILY_DIR,
    ),

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (
                sensor_root /
                group /
                period
            )


            folder_records.append(
                {

                    "sensor":
                        sensor_name,

                    "group":
                        group,

                    "period":
                        period,

                    "folder":
                        str(
                            folder
                        ),

                    "tiff_count":
                        len(
                            list(
                                folder.glob(
                                    "*.tif"
                                )
                            )
                        ),

                }
            )


folder_summary = pd.DataFrame(
    folder_records
)


FOLDER_SUMMARY_FILE = (
    DAILY_DIR /
    "daily_folder_summary.csv"
)


folder_summary.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False,
)


# =============================================================================
# 52. Failure summary
# =============================================================================

if (
    not combined_inventory.empty
    and
    "status"
    in combined_inventory.columns
):

    failed_downloads = (
        combined_inventory
        .loc[
            combined_inventory[
                "status"
            ]
            ==
            "failed"
        ]
        .copy()
    )


else:

    failed_downloads = pd.DataFrame()


print(
    "\nFailed image count:",
    len(
        failed_downloads
    )
)


if not failed_downloads.empty:

    FAILED_DOWNLOAD_FILE = (
        DAILY_DIR /
        "daily_failed_downloads.csv"
    )


    failed_downloads.to_csv(
        FAILED_DOWNLOAD_FILE,
        index=False,
    )


# =============================================================================
# 53. Site coverage
# =============================================================================

if not combined_inventory.empty:

    coverage_check = (
        combined_inventory
        .groupby(
            [
                "sensor",
                "group",
            ],
            as_index=False,
        )
        .agg(

            unique_sites=(
                "site_id",
                "nunique",
            ),

            image_records=(
                "site_id",
                "count",
            ),

        )
    )


    print(
        "\n"
        + "=" * 90
    )


    print(
        "SITE COVERAGE"
    )


    print(
        "=" * 90
    )


    print(
        coverage_check.to_string(
            index=False
        )
    )


# =============================================================================
# 54. Quality summary
# =============================================================================

if not daily_summary.empty:

    print(
        "\n"
        + "=" * 90
    )


    print(
        "IMAGE QUALITY SUMMARY"
    )


    print(
        "=" * 90
    )


    print(
        daily_summary[
            [
                "sensor",
                "group",
                "period",
                "number_of_sites",
                "observed_site_days",
                "mean_valid_pixel_percentage",
                "median_valid_pixel_percentage",
                "images_ge_80pct_valid",
                "percent_images_ge_80pct_valid",
            ]
        ]
        .to_string(
            index=False
        )
    )


# =============================================================================
# 55. Final summary
# =============================================================================

print(
    "\n"
    + "=" * 100
)


print(
    "NOTEBOOK 10 COMPLETE"
)


print(
    "=" * 100
)


print(
    "\nStudy period:"
)


print(
    "2024-05-10 through 2025-02-13"
)


print(
    "\nBefore:"
)


print(
    "2024-05-10 through 2024-09-26"
)


print(
    "\nAfter:"
)


print(
    "2024-09-27 through 2025-02-13"
)


print(
    "\nTreatment sites requested:"
)


print(
    N_TREATMENT_SITES
)


print(
    "\nControls per treatment requested:"
)


print(
    N_CONTROLS_PER_TREATMENT
)


print(
    "\nTreatment sites actually selected:"
)


print(
    len(
        treatment_to_process
    )
)


print(
    "\nCounterfactual sites actually selected:"
)


print(
    len(
        counterfactual_to_process
    )
)


print(
    "\nMaster acquisition-level folder:"
)


print(
    DAILY_DIR
)


print(
    "\nSelected sample:"
)


print(
    SELECTED_SAMPLE_FILE
)


print(
    "\nSentinel-1 inventory:"
)


print(
    S1_INVENTORY_FILE
)


print(
    "\nSentinel-2 inventory:"
)


print(
    S2_INVENTORY_FILE
)


print(
    "\nCombined inventory:"
)


print(
    COMBINED_INVENTORY_FILE
)


print(
    "\nDaily calendar:"
)


print(
    DAILY_CALENDAR_FILE
)


print(
    "\nDataset summary:"
)


print(
    DAILY_SUMMARY_FILE
)


print(
    "\nFolder summary:"
)


print(
    FOLDER_SUMMARY_FILE
)


print(
    "\nExisting valid TIFFs are reused:"
)


print(
    SKIP_EXISTING
)


print(
    "\nIf the notebook stops, rerun it."
)


print(
    "Previously downloaded valid TIFFs will be skipped."
)


print(
    "\nNotebook completed successfully."
)

Packages loaded successfully.

Study period:
2024-05-10 through 2025-02-13

Before:
2024-05-10 through 2024-09-26

After:
2024-09-27 through 2025-02-13

Daily dataset root:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets

Earth Engine initialized.

All available treatment sites: 26
All available counterfactual sites: 260

Selected treatment sites:
   treatment_0001
   treatment_0002
   treatment_0003
   treatment_0004
   treatment_0005
   treatment_0006
   treatment_0007
   treatment_0008
   treatment_0009
   treatment_0010

Controls selected per treatment:
treatment_site_id
treatment_0001    5
treatment_0002    5
treatment_0003    5
treatment_0004    5
treatment_0005    5
treatment_0006    5
treatment_0007    5
treatment_0008    5
treatment_0009    5
treatment_0010    5

CURRENT SAMPLE

Treatment sites: 10
Counterfactual sites: 50
Total sites: 60

Selected sample saved:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/fina

  0%|                                                      |0/1 tiles [00:00<?]


  18/34 2024-10-02 [after]
    Local TIFF not available — querying Earth Engine.
    Download attempt 1...


  0%|                                                      |0/1 tiles [00:00<?]


  19/34 2024-10-07 [after]
    Local TIFF not available — querying Earth Engine.
    Download attempt 1...


...


  38/39 2025-02-07 [after]
    Local TIFF not available — querying Earth Engine.
    Download attempt 1...


  0%|                                                      |0/1 tiles [00:00<?]


  39/39 2025-02-12 [after]
    Local TIFF not available — querying Earth Engine.
    Download attempt 1...


  0%|                                                      |0/1 tiles [00:00<?]


Sentinel-1 inventory saved:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets/sentinel1_daily_inventory.csv

Combined inventory saved:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets/daily_satellite_inventory.csv

DOWNLOAD / REUSE SUMMARY
 download_action  image_count
skipped_existing         2531
      downloaded         1550

Calendar days: 280

Failed image count: 0

SITE COVERAGE
   sensor          group  unique_sites  image_records
sentinel1 counterfactual            50           1599
sentinel1      treatment            10            336
sentinel2 counterfactual            50           1794
sentinel2      treatment            10            352

IMAGE QUALITY SUMMARY
   sensor          group period  number_of_sites  observed_site_days  mean_valid_pixel_percentage  median_valid_pixel_percentage  images_ge_80pct_valid  percent_images_ge_80pct_valid
sentinel1      treatment before               10    